In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat  # Pour charger les fichiers .mat
import sys
import PySpin
import flir_tools
import flir_video
import flir_image
import SLMcontrol
import Fonctions
import time
from scipy.ndimage import zoom
import threading
from PyQt5.QtWidgets import QApplication, QMainWindow, QLabel, QDesktopWidget, QVBoxLayout, QWidget
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import Qt, QTimer, QObject, pyqtSignal
%matplotlib qt

In [2]:
from scipy.optimize import curve_fit

### Camera FLIR

In [3]:
#Camera initialization

sys, clist, cam = flir_tools.connect_cam()
DXmax = 1280;     # Maximum width of the frame (pixels)
DYmax = 1024;     # Maximum height of the frame (pixels)
Xinmax = 0;       # Horizontal offset (starting X-coordinate)
Yinmax = 0;       # Vertical offset (starting Y-coordinate)
tempsexp = 7;

In [4]:
# Video

tempsexp = 7
flir_video.live_view(cam, tempsexp)

>>> LIVE en cours... Fermez la fenêtre pour ARRETER le processus.

[!] Fenêtre fermée. Interruption volontaire du script.


SystemExit: Script stoppé par l'utilisateur.

c:\Users\Manip_1\anaconda3\envs\flir_env\lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [7]:
# Image

image = flir_image.capture(cam, tempsexp) 

plt.figure()  
plt.imshow(image, cmap='turbo')
plt.colorbar()
plt.title('Image')
plt.show()

In [5]:
# Camera ROI

data = flir_image.capture(cam, tempsexp)

DXff, DYff = 624, 624
Xinff, Yinff = 0, 0

if data is not None:
    h = plt.figure()
    plt.imshow(data, cmap='turbo')
    plt.title("Click on ROI center")
    pts = plt.ginput(1) 
    plt.close(h)
    
    if pts:
        xcenter, ycenter = pts[0]
        Xinff = int(np.floor(xcenter) - np.floor(DXff / 2))
        Yinff = int(np.floor(ycenter) - np.floor(DYff / 2))
        Xinff = max(0, min(Xinff, DXmax - DXff))
        Yinff = max(0, min(Yinff, DYmax - DYff))
        Xinff = 8 * (Xinff // 8)
        Yinff = 8 * (Yinff // 8)
        tempsexp = flir_video.live_view(cam, tempsexp, DXff, DYff, Xinff, Yinff)

>>> LIVE en cours... Fermez la fenêtre pour ARRETER le processus.

[!] Fenêtre fermée. Interruption volontaire du script.


SystemExit: Script stoppé par l'utilisateur.

In [7]:
image = flir_image.capture(cam, tempsexp, DXff, DYff, Xinff, Yinff) 

plt.figure()  
plt.imshow(image, cmap='turbo')
plt.colorbar()
plt.title('Image')
plt.show()

### SLM Holoeye

In [6]:
# SLM Initilization

controller = SLMcontrol.init_slm_display()
slm_width = controller.slm_window.width()
slm_height = controller.slm_window.height()

A0 = np.zeros((slm_height, slm_width), dtype=np.uint8)

In [7]:
# SLM ROI

Cxslm = 300
Dxslm = 500
Cyslm = 400
Dyslm = 500

Nrand = 50
SLMrand = np.floor(np.random.rand(Nrand, Nrand) + 0.5).astype(np.uint8)

SLMrand = Fonctions.calculCentreContour(Cxslm, Dxslm, Cyslm, Dyslm, SLMrand, A0)
SLMrand = (SLMrand.astype(np.uint16) * 100) 
SLMrand = np.clip(SLMrand, 0, 255).astype(np.uint8) 

SLMcontrol.update_slm_image(SLMrand)
time.sleep(0.2)

# 1. SLM calibration

1. Take reference speckle with blank SLM

In [8]:
SLMcontrol.update_slm_image(A0)
reference_data = flir_image.capture(cam, tempsexp, DXff, DYff, Xinff, Yinff)

2. Define the SLM random phase mask

In [9]:
h,w = 20, 20
random_matrix = np.random.rand(h,w)
M0 = (random_matrix > 0.5).astype(np.uint8)
M_display = A0.copy() # don't modify A0
M = Fonctions.calculCentreContour(960, 1900, 540, 1060, M0, M_display)

3. SLM calibration loop

In [10]:
vec, C = np.arange(0,256, 5), []

for k in range(len(vec)):
    M1 = M*vec[k]
    SLMcontrol.update_slm_image(M1)
    QApplication.processEvents()
    time.sleep(0.3)
    data = flir_image.capture(cam, tempsexp, DXff, DYff, Xinff, Yinff)
    C.append(Fonctions.corr2(data, reference_data))
    
plt.figure()
plt.plot(vec, C, 'r+') # 'r+-' : rouge, points en croix, reliés
plt.xlabel('Valeur vec[k]')
plt.ylabel('Corrélation')
plt.title('Corrélation')
plt.grid(True)
plt.show()

----------------------------------------------------------------------------------------------------------

Find $x_{\pi}$ and $x_{2\pi}$ :

In [19]:
value_pi, value_2pi = min(C), max(C)
x0, xpi, x2pi = 0, vec[C.index(value_pi)], vec[C.index(value_2pi)]
print(f"x0 = {x0}, xpi = {xpi} and x2pi = {x2pi}")

x0 = 0, xpi = 145 and x2pi = 255


Cosine fit:

In [33]:
def fit_cosine(data):
    # 1. Préparation des axes
    y = np.array(data)
    x = np.arange(len(y))

    # 2. Estimation automatique des paramètres de départ (p0)
    amp_guess = (np.max(y) - np.min(y)) / 2
    off_guess = np.mean(y)
    # On estime la fréquence via la transformée de Fourier rapide (FFT)
    fft_data = np.abs(np.fft.fft(y - off_guess))
    freq_guess = np.fft.fftfreq(len(y))[np.argmax(fft_data[1:]) + 1]

    # p0 = [Amplitude, Fréquence, Phase, Offset]
    p0 = [amp_guess, freq_guess, 0, off_guess]

    # 3. Définition du modèle mathématique
    def model(x, a, f, p, o):
        return a * np.cos(2 * np.pi * f * x + p) + o

    # 4. Calcul du Fit
    popt, _ = curve_fit(model, x, y, p0=p0)
    
    # 5. Génération de la courbe fittée
    y_fit = model(x, *popt)
    
    return x, y_fit, popt

x, y_fit, params = fit_cosine(C)
yi = (y_fit - params[3]) / params[0]
xi = x

plt.plot(C, 'b.', label='Données', alpha=0.5)
plt.plot(x, y_fit, 'r-', label='Fit Cosinus', lw=2)
plt.legend()
plt.show()

Get pixtorad and radtopix functions:


In [36]:
x0 = np.argmax(yi[:len(yi)//2]) 
xpi = np.argmin(yi)
x2pi = np.argmax(yi[xpi:]) + xpi

Xi = np.arange(int(x0), int(x2pi) + 1)
yi_segment = yi[int(x0) : int(x2pi) + 1]
Yi = np.zeros(len(yi_segment))

k_split = int(xpi - x0)

# De 0 à pi
Yi[0 : k_split + 1] = np.arccos(np.clip(yi_segment[0 : k_split + 1], -1, 1))

# De pi à 2pi
Yi[k_split + 1 :] = -np.arccos(np.clip(yi_segment[k_split + 1 :], -1, 1)) + 2 * np.pi

plt.figure()
plt.plot(Xi, Yi, '*')
plt.title("Phase extraite (Yi) à partir du fit")
plt.show()

In [37]:
# 1. Points de passage obligatoires
px1, py1 = 0, 0
px2, py2 = x2pi, 2 * np.pi

# 2. Calcul de la droite de base 
pente = (py2 - py1) / (px2 - px1)
intercept = py1 - pente * px1

# 3. 
def model_contraint(x, a, b):
    return (x - px1) * (x - px2) * (a * x + b) + (pente * x + intercept)

# 4. Préparation des données et fit
xData = np.array(Xi)
yData = np.array(Yi)

# Fit non-linéaire 
popt, _ = curve_fit(model_contraint, xData, yData, p0=[0.1, 0.1])

# A(1) et A(2) 
A1, A2 = popt

# 5. Modèle final pour affichage
x_plot = np.arange(0, x2pi + 1)
y_plot = model_contraint(x_plot, A1, A2)

# --- Affichage ---
plt.figure(figsize=(10, 6))
plt.plot(xData, yData, '+', label='Mesures (Xi, Yi)')
plt.plot(x_plot, y_plot, 'r-', label='Fit contraint')
plt.plot([px1, px2], [py1, py2], 'ro', markersize=10, label='Points imposés')

plt.title('Fit Polynomial Contraint (0,0) et (x2pi, 2pi)')
plt.xlabel('Niveau de gris')
plt.ylabel('Phase (rad)')
plt.legend()
plt.grid(True)
plt.show()

print(f"Parameter A(1): {A1:e}")
print(f"Parameter A(2): {A2:e}")

Parameter A(1): -3.360671e-05
Parameter A(2): 1.565087e-03


In [45]:
xphi = np.arange(0, x2pi + 1)
phi = Fonctions.PixelSLM_pixversrad_Holoeyenir80(xphi)

plt.figure()
plt.plot(xphi, phi, label='Modèle')
plt.plot(Xi, Yi, '+', label='Mesures')
plt.xlabel('xphi')
plt.ylabel('phi')
plt.legend()
plt.show()

AttributeError: module 'Fonctions' has no attribute 'PixelSLM_pixversrad_Holoeyenir80'

# 2. SLM optimization

In [22]:
# --- 1. Settings ---
Nseg = 4
numIterations = 4 * Nseg * Nseg
Nmoy_ps = 3
Ntotal = Nseg**2

current_phases = np.zeros(Ntotal)
I_max_history = np.zeros(numIterations)

Nphase = 4
alpha_fit = np.linspace(0, 2*np.pi, Nphase, endpoint=False)
datalin = np.zeros(Nphase)

# --- 2. Figure Setup ---
plt.ion()
fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(1, 3, 1)
h_fit_line, = ax1.plot([], [], '-b', lw=2)
ax1.set_xlim(0, numIterations)
ax1.set_ylim(0, 255) # Ajuster selon intensité max
ax1.grid(True)

ax2 = fig.add_subplot(1, 3, 2)
zoom_size = 30
h_live = ax2.imshow(np.zeros((zoom_size*2+1, zoom_size*2+1)), cmap='turbo')
ax2.plot(zoom_size, zoom_size, 'wx', markersize=10)

ax3 = fig.add_subplot(1, 3, 3)
h_slm_map = ax3.imshow(np.zeros((Nseg, Nseg)), cmap='turbo')

# --- 3. Optimization Loop ---
for it in range(numIterations):
    idx_to_optimize = np.random.permutation(Ntotal)[:int(Ntotal/2)]
    
    # Step A: Measure N intensities
    for k in range(Nphase):
        temp_phases = current_phases.copy()
        temp_phases[idx_to_optimize] = (temp_phases[idx_to_optimize] + alpha_fit[k]) % (2*np.pi)
        
        grey_values = np.array([PixelSLM_radverspix_Holoeyenir80_Theo(p) for p in temp_phases])
        small_map = grey_values.reshape((Nseg, Nseg))
        
        M = Fonctions.calculCentreContour(Cxslm, Dxslm, Cyslm, Dyslm, small_map, A0)
        # Affichage du masque sur le SLM via ta fonction dédiée
        # SLMcontrol.update(M) 
        
        time.sleep(0.15)
        
        tmp_stack = np.zeros(Nmoy_ps)
        for m in range(Nmoy_ps):
            img_calc = flir_image.capture(cam, tempsexp, DXff, DYff, Xinff, Yinff)
            tmp_stack[m] = img_calc[y_t, x_t]
        
        datalin[k] = np.mean(tmp_stack)
    
    # --- Step B: Phasor Formula ---
    datalin_clean = datalin - np.mean(datalin)
    complex_sum = np.sum(datalin_clean * (np.cos(alpha_fit) + 1j*np.sin(alpha_fit)))
    phi_optimal = np.angle(complex_sum) % (2*np.pi)
    
    # Update
    current_phases[idx_to_optimize] = (current_phases[idx_to_optimize] + phi_optimal) % (2*np.pi)
    
    # --- Step C: MEASURE PHYSICAL I_MAX ---
    final_grey = np.array([PixelSLM_radverspix_Holoeyenir80_Theo(p) for p in current_phases])
    opt_map = final_grey.reshape((Nseg, Nseg))
    h_slm_map.set_data(opt_map)
    
    M_opt = Fonctions.calculCentreContour(Cxslm, Dxslm, Cyslm, Dyslm, opt_map, A0)
    # SLMcontrol.update(M_opt)
    
    time.sleep(0.15)
    
    tmp_final = np.zeros(Nmoy_ps)
    for m in range(Nmoy_ps):
        img_live = flir_image.capture(cam, tempsexp, DXff, DYff, Xinff, Yinff)
        
        y_low, y_high = max(0, y_t-zoom_size), min(img_live.shape[0], y_t+zoom_size+1)
        x_low, x_high = max(0, x_t-zoom_size), min(img_live.shape[1], x_t+zoom_size+1)
        
        h_live.set_data(img_live[y_low:y_high, x_low:x_high])
        tmp_final[m] = img_live[y_t, x_t]
    
    I_max_history[it] = np.mean(tmp_final)
    h_fit_line.set_xdata(np.arange(it + 1))
    h_fit_line.set_ydata(I_max_history[:it + 1])
    
    plt.draw()
    plt.pause(0.01)


NameError: name 'PixelSLM_radverspix_Holoeyenir80_Theo' is not defined